In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image  # retained for compatibility, not used in new loader
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import tensorflow as tf
import concurrent.futures
from functools import partial

# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)



2026-05-07 04:18:20.656694: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778127500.672231      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778127500.676934      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-07 04:18:20.697448: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
COMPETITION_NAME = "ranzcr-clip-catheter-line-classification"
base_path = f"/kaggle/input/{COMPETITION_NAME}"
data_dir = base_path

train_csv_path = os.path.join(data_dir, "train.csv")
sample_submission_path = os.path.join(data_dir, "sample_submission.csv")

train_df = pd.read_csv(train_csv_path)
sub_df = pd.read_csv(sample_submission_path)

orig_label_cols = [c for c in sub_df.columns if c != "StudyInstanceUID"]
label_cols = [c for c in train_df.columns if c not in ("StudyInstanceUID", "PatientID")]
num_labels = len(label_cols)

missing_labels = set(label_cols) - set(sub_df.columns)
for col in missing_labels:
    sub_df[col] = 0.0

sub_df = sub_df[["StudyInstanceUID"] + label_cols]




In [3]:
def _load_image_tf(path):
    """TensorFlow image loader: read, decode, resize, normalize, flatten."""
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, [32, 32])
    img = img / 255.0  # scale to [0,1]
    return tf.reshape(img, [-1])  # flatten to 1‑D


def compute_simple_features(uids, img_dir):
    """
    Fast feature extraction using tf.data pipeline.
    Returns a (len(uids), 32*32*3) float32 NumPy array.
    """
    # Build full file paths
    paths = [os.path.join(img_dir, f"{uid}.jpg") for uid in uids]

    # Create TensorFlow dataset and map loading function with parallelism
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(_load_image_tf, num_parallel_calls=tf.data.AUTOTUNE)

    # Materialize to NumPy – tf.data will pipeline reads efficiently
    feats = np.stack(list(ds.as_numpy_iterator()))
    return feats.astype(np.float32)




In [4]:
train_img_dir = os.path.join(data_dir, "train")
train_uids = train_df["StudyInstanceUID"].values
X_train = compute_simple_features(train_uids, train_img_dir)

y_train = train_df[label_cols].values.astype(np.float32)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train[:, 0]
)

base_clf = LogisticRegression(
    penalty="l2",
    C=4.0,
    solver="saga",
    max_iter=200,
    n_jobs=1,  # avoid oversubscription; outer OvR will parallelise.
    class_weight="balanced",
)

scaled_clf = make_pipeline(StandardScaler(with_mean=False), base_clf)

model = OneVsRestClassifier(scaled_clf, n_jobs=-1)

model.fit(X_tr, y_tr)

val_preds = model.predict_proba(X_val)
auc_scores = []
for i in range(num_labels):
    try:
        auc = roc_auc_score(y_val[:, i], val_preds[:, i])
    except ValueError:
        auc = np.nan
    auc_scores.append(auc)
print("Validation AUC per label:", auc_scores)
print("Mean validation AUC:", np.nanmean(auc_scores))



2026-05-07 04:18:27.329398: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778127507.329859      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:05:00.0, compute capability: 8.6
2026-05-07 04:19:52.056759: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/line

Validation AUC per label: [0.6154259259259259, 0.7662247838616716, 0.8330517528247265, 0.7141290520682547, 0.6707624370041415, 0.799067225430177, 0.8094345561051687, 0.5342850267251369, 0.5508871582662931, 0.5376394294023503, 0.7874361980962892]
Mean validation AUC: 0.6925766859736486


In [5]:
test_img_dir = os.path.join(data_dir, "test")
test_uids = sub_df["StudyInstanceUID"].values
X_test = compute_simple_features(test_uids, test_img_dir)

test_preds = model.predict_proba(X_test)

for idx, col in enumerate(label_cols):
    if col in orig_label_cols:
        sub_df[col] = test_preds[:, idx].astype(np.float32)



2026-05-07 04:25:17.092947: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [6]:
submission_path = "submission.csv"
final_submission = sub_df[["StudyInstanceUID"] + orig_label_cols]
final_submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")
final_submission.head()

Submission written to submission.csv


,StudyInstanceUID,ETT - Abnormal,ETT - Borderline,ETT - Normal,NGT - Abnormal,NGT - Borderline,NGT - Incompletely Imaged,NGT - Normal,CVC - Abnormal,CVC - Borderline
0,1.2.826.0.1.3680043.8.498.25512976433640891933...,0.0,0.041945,0.759479,2.209674e-04,0.004866,0.016437,0.553188,0.155693,0.471773
1,1.2.826.0.1.3680043.8.498.24449897997512078380...,0.0,0.017012,0.061334,1.754692e-06,0.011443,0.061654,0.067284,0.586558,0.600901
2,1.2.826.0.1.3680043.8.498.38485493636649999035...,0.0,0.010871,0.651762,1.028895e-10,0.000005,0.543050,0.057826,0.679841,0.256164
3,1.2.826.0.1.3680043.8.498.11901773728604504629...,0.0,0.194240,0.211797,4.762765e-09,0.001816,0.092183,0.350228,0.594183,0.470612
4,1.2.826.0.1.3680043.8.498.11327663178389439022...,0.0,0.248397,0.514671,2.517415e-08,0.709889,0.293068,0.902548,0.846612,0.666472
